# 03 Tuned XGBoost

Use this notebook only for tuned XGBoost experiments, metric tracking, and exporting the best model artifacts.

In [1]:
import json
import os
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

sys.path.append(os.path.abspath('..'))

from src.training.evaluate import evaluate_regression_model

In [4]:
feature_df = pd.read_csv(Path('../data/processed/california_housing_features.csv'))
feature_metadata = json.loads(Path('../data/processed/california_housing_feature_metadata.json').read_text())
X = feature_df.drop(columns=['median_house_value'])
y = feature_df['median_house_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
best_params = {
    'subsample': 1.0,
    'reg_lambda': 3,
    'reg_alpha': 1,
    'n_estimators': 900,
    'min_child_weight': 7,
    'max_depth': 5,
    'learning_rate': 0.07,
    'colsample_bytree': 0.7,
}
best_params

{'subsample': 1.0,
 'reg_lambda': 3,
 'reg_alpha': 1,
 'n_estimators': 900,
 'min_child_weight': 7,
 'max_depth': 5,
 'learning_rate': 0.07,
 'colsample_bytree': 0.7}

In [6]:
model = XGBRegressor(
    **best_params,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = evaluate_regression_model(y_test, predictions)
metrics

{'rmse': 0.4369736776555384,
 'mae': 0.28467240183382303,
 'r2': 0.8542851902463815}

In [7]:
feature_payload = {
    'model_name': 'xgboost',
    'feature_columns': X.columns.tolist(),
    'clip_thresholds': feature_metadata['clip_thresholds'],
    'feature_metadata_path': '../data/processed/california_housing_feature_metadata.json',
}
joblib.dump(model, '../models/xgboost_price_model_tuned_clean.joblib')
Path('../models/xgboost_price_model_features.json').write_text(json.dumps(feature_payload, indent=2))
Path('../models/xgboost_price_model_metrics.json').write_text(json.dumps(metrics, indent=2))
'saved'

'saved'